# 05 — WO-2 Analytical Investigation

## Purpose

This notebook orchestrates the 21 canonical WO-2 SQL investigations across AN0–AN4. It preserves the separation between SQL definitions, shared database execution, and analytical interpretation:

- Canonical SQL files live in `sql/04_analytics/`.
- `src/analytics/db.py` owns the read-only connection and query execution.
- This notebook runs query IDs, displays every returned result set, and records interpretation after evidence is reviewed.

This is the single WO-2 notebook. The historical `04_sql_analytics.ipynb` remains the WO-1 evidence record and is not modified.

## Execution status and safety

**No WO-2 query has been executed by preparing this notebook.** The live execution gate below defaults to `False`. It must remain disabled until a separate explicit approval to query the project database is given.

Do not use **Run All**. Execute one query cell at a time and stop at each review gate. AN3.06 scans the large `staging.student_vle` table and must be run alone. All SQL is expected to run in a read-only session; this notebook does not alter RAW, QUALITY, or STAGING and creates no database objects.

## Work-order sequence

1. WO2-A: AN0.07, review, then AN1.10.
2. WO2-B: AN2.10, review, then AN3.01–AN3.03.
3. WO2-C: AN3.04–AN3.08; execute AN3.06 separately.
4. WO2-D: AN4.01, AN4.02, AN4.04, AN4.05.
5. WO2-E: AN4.06 first, then AN4.03, followed by AN4.07–AN4.10.
6. WO2-F: evidence review and documentation closeout.

The WO2-A/B/C/D/E labels are execution packages, not replacements for official analytical identifiers AN0–AN11.

## 1. Environment and controlled execution

The next cell resolves the repository root, imports the shared runner, and defines a bounded display helper. It does not open a database connection. Every result set is displayed independently with its statement number, visible row count, and truncation flag. The runner's `elapsed_seconds` is the total SQL-file batch duration, not a per-statement timing.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "analytics" / "db.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "analytics" / "db.py").exists():
    raise FileNotFoundError("Run Jupyter from the repository root or its notebooks directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from src.analytics.db import run_query

# Keep False until a separate explicit approval for live WO-2 database execution.
LIVE_WO2_EXECUTION_AUTHORIZED = False

# Review flags are changed manually only after inspecting the actual returned outputs.
AN0_07_REVIEWED = False
AN1_10_REVIEWED = False
AN2_10_REVIEWED = False
AN3_REVIEWED = False
AN4_D_REVIEWED = False
AN4_06_REVIEWED = False

query_results = {}

def execute_query(query_id, *, statement_timeout_ms=120000, requires=None):
    if not LIVE_WO2_EXECUTION_AUTHORIZED:
        raise PermissionError(
            "Live WO-2 execution is disabled. Obtain explicit approval, then set "
            "LIVE_WO2_EXECUTION_AUTHORIZED = True in this notebook."
        )
    if requires and not globals().get(requires, False):
        raise RuntimeError(
            f"Stop gate: review checkpoint {requires} must be set to True after "
            "examining the preceding query outputs."
        )

    result = run_query(
        query_id,
        max_rows=500,
        statement_timeout_ms=statement_timeout_ms,
    )
    query_results[query_id] = result

    print(
        f"{query_id} | {result['query_file']} | "
        f"{result['result_set_count']} result set(s) | "
        f"batch elapsed: {result['elapsed_seconds']} s | "
        f"timeout: {result['statement_timeout_ms']} ms"
    )
    for result_set in result["result_sets"]:
        print(
            f"Result set {result_set['statement_number']}: "
            f"{result_set['rows_displayed']} displayed row(s); "
            f"truncated={result_set['display_truncated']}"
        )
        display(result_set["data"])

    return result

## 2. WO2-A — Structural verification and join fan-out

### AN0.07 — RAW primary-key and index verification

This catalog-only query returns three result sets: documented-versus-actual primary keys, RAW constraint counts by type, and every index on `raw.student_vle`. A non-unique candidate key does not imply that no index exists.

**Review gate:** inspect all three result sets. Confirm the six documented primary keys and expected absence of a primary key on `student_vle`; investigate any mismatch before proceeding.

In [ ]:
an0_07 = execute_query("AN0.07")

### AN1.10 — Assessment join fan-out decomposition

This query returns three result sets and decomposes the historical increase observed when joining `student_assessment` to `student_info` by student ID alone. The expected fan-out formula depends on one row per student-presentation in `student_info`; AN0.07 is not sufficient by itself to validate that grain, so the existing AN1.01 evidence remains a dependency. The historical 33,407 excess rows are a reference, not a forced result.

**Review gate:** inspect all three outputs and require reconciliation, zero submissions without student information, and zero unmatched rows on the correct join path. Do not proceed if any condition fails.

In [ ]:
if not AN0_07_REVIEWED:
    raise RuntimeError("Review AN0.07's three result sets and set AN0_07_REVIEWED = True before running AN1.10.")
an1_10 = execute_query("AN1.10", requires="AN0_07_REVIEWED")

## 3. WO2-B — IMD missingness and initial withdrawal investigation

### AN2.10 — IMD missingness extension

This query returns three result sets: missingness by module, every region/module/presentation group with missing IMD (including NULL region), and cross-presentation alternative-value classifications. Denominators remain the complete relevant groups. Alternative values are descriptive only and do not authorize imputation.

**Review gate:** reconcile the total missing rows with the historical 1,111-row reference and inspect both row and distinct-student counts.

In [ ]:
if not AN1_10_REVIEWED:
    raise RuntimeError("Review AN1.10 and set AN1_10_REVIEWED = True before running AN2.10.")
an2_10 = execute_query("AN2.10", requires="AN1_10_REVIEWED")

### AN3.01 — Final result versus unregistration

Cross-tabulate `final_result` with the presence of `date_unregistration`. These fields are not interchangeable; inspect both off-diagonal populations.

In [ ]:
an3_01 = execute_query("AN3.01", requires="AN2_10_REVIEWED")

### AN3.02 — Withdrawal definition conflicts

Identify mismatches between the Withdrawn outcome and unregistration-date evidence. Review the timing relative to presentation length before defining any withdrawal rule.

In [ ]:
an3_02 = execute_query("AN3.02", requires="AN2_10_REVIEWED")

### AN3.03 — Withdrawal timing distribution

Classify unregistration days relative to presentation start and course end. The after-course-end test takes precedence over week buckets so a short presentation is not misclassified as having in-course unregistrations after its actual end.

In [ ]:
an3_03 = execute_query("AN3.03", requires="AN2_10_REVIEWED")

## 4. WO2-C — Registration timing and withdrawal-related activity

### AN3.04 — Registration timing profile

Report missing registration dates separately and compare withdrawal proportions by registration-time bucket. These are descriptive associations, not causal effects.

In [ ]:
an3_04 = execute_query("AN3.04", requires="AN2_10_REVIEWED")

### AN3.05 — Registration and unregistration order

Classify missing, reversed, same-day, and later registration/unregistration sequences. Do not convert unusual sequences into source corrections.

In [ ]:
an3_05 = execute_query("AN3.05", requires="AN2_10_REVIEWED")

### AN3.06 — Withdrawn student-presentations and recorded VLE activity

**High-runtime query. Run this cell alone.** It aggregates `staging.student_vle` to student-presentation grain before joining. A `NO_RECORDED_ACTIVITY_ROW` classification means no matching row was found in the stored VLE activity table; it does not prove no learning or participation. `NO_REGISTRATION_MATCH` is kept distinct from a matched registration with a NULL unregistration date.

The runner timeout is passed as an argument; do not add SQL `SET` statements. Review the output before running other queries.

In [ ]:
an3_06 = execute_query(
    "AN3.06",
    statement_timeout_ms=600000,
    requires="AN2_10_REVIEWED",
)

### AN3.07 — Outcome rates by previous attempts and education

Compare outcome shares across the two student attributes. The highest previous-attempt category is grouped as `3+`; categories below 100 rows are flagged. Repeated students may appear across presentations, so observations are not independent people.

In [ ]:
an3_07 = execute_query("AN3.07", requires="AN2_10_REVIEWED")

### AN3.08 — Withdrawal rate by presentation

Report withdrawn and pass/distinction shares with the student-presentation denominator. Do not use these descriptive rates as a presentation-quality ranking.

In [ ]:
an3_08 = execute_query("AN3.08", requires="AN2_10_REVIEWED")

**WO2-C review checkpoint:** after reviewing AN3.01–AN3.08, record the withdrawal definition, date limitations, denominator interpretation, and whether any anomalies require follow-up. Only then set `AN3_REVIEWED = True`.

## 5. WO2-D — Assessment structure, dates, and scores

### AN4.01 — Assessment structure by presentation

Describe assessment counts, types, weights, and due-date coverage by module and presentation. Do not assume that any type must exist or that weights must sum to 100.

In [ ]:
an4_01 = execute_query("AN4.01", requires="AN3_REVIEWED")

### AN4.02 — Assessments with missing dates

Profile every assessment with a missing due date and its linked submission count. Assessment type is observed, not assumed.

In [ ]:
an4_02 = execute_query("AN4.02", requires="AN3_REVIEWED")

### AN4.04 — Submissions without a due date

Report submission rows linked to assessments with NULL due dates, retaining module, presentation, type, and banked status. Do not infer lateness without a valid due-date reference.

In [ ]:
an4_04 = execute_query("AN4.04", requires="AN3_REVIEWED")

### AN4.05 — Score profile and NULL scores

This query returns two result sets: score distribution by assessment type and banked status, then assessment-level locations of NULL scores. Keep NULL distinct from numeric zero.

In [ ]:
an4_05 = execute_query("AN4.05", requires="AN3_REVIEWED")

**WO2-D review checkpoint:** reconcile assessment and submission counts, inspect missing dates and NULL scores, and document the observed assessment-type and banked-status representations. Only then set `AN4_D_REVIEWED = True`.

## 6. WO2-E — Banked assessments, timing, participation, and coverage

### AN4.06 — Banked assessment profile (run first)

Inspect the stored `is_banked` values, including NULL, by assessment type. This is a descriptive profile only; it does not decide whether banked rows should be excluded.

In [ ]:
an4_06 = execute_query("AN4.06", requires="AN4_D_REVIEWED")

After reviewing AN4.06 and checking the source semantics, set `AN4_06_REVIEWED = True`. This gate is required before AN4.03 and the remaining banked-status analyses.

### AN4.03 — Submission timing versus due date

Classify submissions before, on, and after due dates, with missing submission dates and banked status retained. Assessments with NULL due dates are handled separately by AN4.04.

In [ ]:
an4_03 = execute_query("AN4.03", requires="AN4_06_REVIEWED")

### AN4.07 — Assessment participation by outcome

Count assessment submission rows per student-presentation and final result. Preserve banked-status groups and explicitly retain student-presentations with no submission rows. A student-presentation may occur in more than one banked-status group.

In [ ]:
an4_07 = execute_query("AN4.07", requires="AN4_06_REVIEWED")

### AN4.08 — Score by final result

Describe non-NULL score distributions by final result, assessment type, and banked status. Distinct counts are at student-presentation grain. Scores may be mechanically related to final outcomes and are potential target leakage; this is not a predictive feature set.

In [ ]:
an4_08 = execute_query("AN4.08", requires="AN4_06_REVIEWED")

### AN4.09 — Submissions after unregistration

Compare submission and unregistration day values. Keep missing registration matches, missing unregistration dates, and missing submission dates separate. The comparison does not prove that a submission was accepted after withdrawal.

In [ ]:
an4_09 = execute_query("AN4.09", requires="AN4_06_REVIEWED")

### AN4.10 — Weighted-score grain and coverage check

This query returns multiple result sets to test the expected assessment-submission key and row grain before summarizing submitted assessment-weight coverage by banked status. It retains all statuses and collapses physical repeats only at the explicitly stated student-assessment-status combination. A student may appear in more than one status group.

The output's average and median submitted-weight shares are calculated among student-presentations with that status, not across the full presentation denominator. The full student-presentation count is shown separately for context. This is not approval of a weighted-score KPI or final-score rule.

In [ ]:
an4_10 = execute_query("AN4.10", requires="AN4_06_REVIEWED")

## 7. WO2-F — Evidence review and closeout

No execution status is inferred from the existence of a query cell. For each query, retain the actual returned outputs in `query_results` and record:

- Execution status, runtime, result-set count, displayed row counts, and truncation.
- Output grain, join cardinality, unmatched rows, and denominator.
- Missing-value treatment and time basis.
- Distinct student versus student-presentation counts.
- Banked-status treatment and any overlap between groups.
- Material limitations, alternative explanations, and decisions.
- The next justified work based on observed evidence.

### Interpretation record

Use this table after the relevant queries have actually run. Leave values blank until supported by outputs.

| Query ID | Execution verified? | Key result / denominator | Interpretation and limitation | Decision / follow-up |
|---|---|---|---|---|
| AN0.07–AN4.10 | No — not executed during preparation | | | |

**Exit rule:** WO-2 is not complete until its outputs are reviewed and reconciled, or explicitly deferred with a documented reason. WO-2 does not authorize AN5, KPI approval, marts, Power BI, predictive modelling, or a separate data-science phase.